In [ ]:
import pandas as pd


In [4]:
gigi = pd.read_parquet("/home/renku/work/store/parquet_data/HackDays2026 - GIGI.parquet")

test = pd.read_parquet("/home/renku/work/store/parquet_data/2024/April 2024/LG_AIM2Hackerdays_kWh_20260812_070837.parquet")


In [3]:
gigi.head()

,GP-Nr,PLZ,Ort,Kanton,WärmePumpe,PV,PV-Leistung in kWp,Batterie/Speicher,Ladestation für Elektrofahrzeuge,Wärmepumpenboiler,Datum Unterschrift,geplanter Baustart,Übergabe,InBetrieb-Datum
0,698970,5452.0,Oberrohrdorf,AG,-,x,NaN,-,-,-,18.04.2008,NaN,NaN,NaN
1,NaN,5024.0,Küttigen,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,NaN
2,196344,5732.0,Zetzwil,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,21.06.2017
3,570021,5023.0,Biberstein,AG,x,-,NaN,-,-,-,01.07.2017,NaN,NaN,24.11.2017
4,716671,8962.0,Bergdietikon,AG,x,x,NaN,x,-,-,24.08.2017,NaN,23.11.2017,04.12.2017


In [5]:
test.head()

,MP ID,OBIS-Code,Datum,PLZ,00:15,00:30,00:45,01:00,01:15,01:30,...,22:00,22:15,22:30,22:45,23:00,23:15,23:30,23:45,00:00,Unnamed: 100
0,53628,1-1:2.29.0*255,01.04.2024,5105.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,53628,1-1:2.29.0*255,02.04.2024,5105.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,53628,1-1:2.29.0*255,03.04.2024,5105.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,53628,1-1:2.29.0*255,04.04.2024,5105.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,53628,1-1:2.29.0*255,05.04.2024,5105.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [23]:
import pandas as pd

# Load lookup tables
mpid_map = pd.read_parquet(
    "../../store/parquet_data/mpid_zähler_mapping.parquet"
)

zaehler_gp = pd.read_parquet(
    "../../store/parquet_data/Zähler-GP.parquet"
)

gigi = pd.read_parquet(
    "../../store/parquet_data/HackDays2026 - GIGI.parquet"
)

# Clean column names
mpid_map.columns = mpid_map.columns.str.strip()
zaehler_gp.columns = zaehler_gp.columns.str.strip()
gigi.columns = gigi.columns.str.strip()

# Normalize IDs
mpid_map["MP ID"] = pd.to_numeric(
    mpid_map["MP ID"], errors="coerce"
).astype("Int64")

zaehler_gp["GPartner"] = pd.to_numeric(
    zaehler_gp["GPartner"], errors="coerce"
).astype("Int64")

gigi["GP-Nr"] = pd.to_numeric(
    gigi["GP-Nr"], errors="coerce"
).astype("Int64")

binary_cols = {
    "WärmePumpe": "has_heatpump",
    "PV": "has_pv",
    "Batterie/Speicher": "has_battery",
    "Ladestation für Elektrofahrzeuge": "has_ev_charger",
    "Wärmepumpenboiler": "has_heatpump_boiler",
}

for source_col, new_col in binary_cols.items():
    cleaned = (
        gigi[source_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    gigi[new_col] = cleaned.map({
        "x": True,
        "-": False
    })

lookup = (
    mpid_map
    .merge(
        zaehler_gp[["Zählpunktbezeichnung", "GPartner"]],
        on="Zählpunktbezeichnung",
        how="left"
    )
    .merge(
        gigi[
            [
                "GP-Nr",
                "PLZ",
                "Ort",
                "Kanton",
                "PV",
                "has_pv",
                "has_heatpump",
                "has_battery",
                "has_ev_charger",
                "has_heatpump_boiler",
            ]
        ],
        left_on="GPartner",
        right_on="GP-Nr",
        how="left"
    )
)

In [16]:
gp_from_zaehler = set(
    zaehler_gp["GPartner"]
    .dropna()
    .astype("Int64")
)

gp_from_gigi = set(
    gigi["GP-Nr"]
    .dropna()
    .astype("Int64")
)

overlap = gp_from_zaehler & gp_from_gigi

print("Unique GPartner:", len(gp_from_zaehler))
print("Unique GP-Nr:", len(gp_from_gigi))
print("Overlap:", len(overlap))
print("Example overlaps:", list(overlap)[:20])

Unique GPartner: 76718
Unique GP-Nr: 877
Overlap: 337
Example overlaps: [np.int64(821251), np.int64(821253), np.int64(821258), np.int64(722955), np.int64(749578), np.int64(821264), np.int64(821268), np.int64(729113), np.int64(186396), np.int64(104479), np.int64(143397), np.int64(161857), np.int64(737347), np.int64(720967), np.int64(727135), np.int64(692320), np.int64(727137), np.int64(163954), np.int64(135287), np.int64(690299)]


In [27]:
matched_only = lookup[lookup["GP-Nr"].notna()].copy()

print("Matched rows:", len(matched_only))
print("Matched unique MP IDs:", matched_only["MP ID"].nunique())

matched_only[matched_only.Kanton != "AG"]

Matched rows: 525
Matched unique MP IDs: 413


,MP ID,Zählpunktbezeichnung,GPartner,GP-Nr,PLZ,Ort,Kanton,PV,has_pv,has_heatpump,has_battery,has_ev_charger,has_heatpump_boiler
86554,84982,CH1011701234500000000000000537164,909265,909265,4468.0,Kienberg,SO,-,False,True,False,False,False
